# Titanic Data Preprocessing & Missing Data Handling
## Notebook dành cho Google Colab

Notebook này hướng dẫn các bước tiền xử lý dữ liệu (preprocessing) và xử lý dữ liệu thiếu (missing data) trên bộ dữ liệu Titanic.

**Nội dung:**
1. Import thư viện
2. Upload & Load dữ liệu
3. Khám phá dữ liệu & Xác định Missing Data
4. Trực quan hóa Missing Data
5. Xử lý Missing Data - Phương pháp xóa
6. Xử lý Missing Data - Imputation (Mean/Median/Mode)
7. Xử lý Missing Data - Imputation nâng cao (KNN, Iterative)
8. Mã hóa biến Categorical
9. Feature Scaling
10. Kiểm tra & Xuất dữ liệu

## 1. Import Thư viện

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler, MinMaxScaler

import warnings
warnings.filterwarnings('ignore')

# Cài thư viện missingno để trực quan hóa missing data
!pip install --quiet missingno
import missingno as msno

## 2. Upload & Load Dữ liệu

Có 2 cách để load dữ liệu trên Colab:
- **Cách 1**: Upload file CSV trực tiếp từ máy tính
- **Cách 2**: Mount Google Drive và đọc file từ Drive

In [ ]:
# ===== CÁCH 1: Upload file từ máy tính =====
from google.colab import files

print("Hãy upload file train.csv và test.csv:")
uploaded = files.upload()

# Load sau khi upload
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')
print(f"\nTrain shape: {train_df.shape}")
print(f"Test shape: {test_df.shape}")

In [ ]:
# ===== CÁCH 2 (thay thế): Mount Google Drive =====
# from google.colab import drive
# drive.mount('/content/drive')
#
# train_df = pd.read_csv('/content/drive/MyDrive/path/to/train.csv')
# test_df = pd.read_csv('/content/drive/MyDrive/path/to/test.csv')

## 3. Khám phá Dữ liệu & Xác định Missing Data

Trước khi xử lý, ta cần hiểu cấu trúc dữ liệu và xác định các cột có giá trị thiếu.

In [ ]:
# Xem vài dòng đầu
print("===== TRAIN DATA =====")
display(train_df.head())

# Cấu trúc dữ liệu
print("\n===== THÔNG TIN DATASET =====")
print(f"Shape: {train_df.shape}")
train_df.info()

In [ ]:
# Thống kê mô tả
train_df.describe()

In [ ]:
# ===== XÁC ĐỊNH MISSING DATA =====
print("===== MISSING DATA TRONG TRAIN =====")
missing_count = train_df.isnull().sum()
missing_pct = train_df.isnull().mean() * 100

missing_info = pd.DataFrame({
    'Số lượng thiếu': missing_count,
    'Phần trăm (%)': missing_pct.round(2)
})
# Chỉ hiện các cột có missing
print(missing_info[missing_info['Số lượng thiếu'] > 0])

print("\n===== MISSING DATA TRONG TEST =====")
missing_count_test = test_df.isnull().sum()
missing_pct_test = test_df.isnull().mean() * 100
missing_info_test = pd.DataFrame({
    'Số lượng thiếu': missing_count_test,
    'Phần trăm (%)': missing_pct_test.round(2)
})
print(missing_info_test[missing_info_test['Số lượng thiếu'] > 0])

## 4. Trực quan hóa Missing Data

Sử dụng `seaborn heatmap` và thư viện `missingno` để nhìn rõ pattern dữ liệu thiếu.

In [ ]:
# Heatmap missing data bằng seaborn
plt.figure(figsize=(12, 6))
sns.heatmap(train_df.isnull(), cbar=True, yticklabels=False, cmap='viridis')
plt.title('Heatmap Missing Data - Train Set')
plt.show()

In [ ]:
# Sử dụng missingno để trực quan hóa
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Matrix plot - cho thấy vị trí missing
msno.matrix(train_df, ax=axes[0], sparkline=False, fontsize=10)
axes[0].set_title('Missing Data Matrix')

# Bar plot - số lượng non-null mỗi cột
msno.bar(train_df, ax=axes[1], fontsize=10)
axes[1].set_title('Missing Data Bar Chart')

plt.tight_layout()
plt.show()

## 5. Xử lý Missing Data - Phương pháp Xóa (Removal)

**Nguyên tắc:**
- Cột có **>70% missing** → nên **xóa cột** (vì impute sẽ không chính xác)
- Dòng có quá nhiều missing → có thể xóa dòng

Trong Titanic dataset:
- `Cabin` có **~77% missing** → **Xóa cột**
- `Age` có **~20% missing** → Sẽ **impute** (ở bước sau)
- `Embarked` có **<1% missing** → Sẽ **impute** (ở bước sau)

In [ ]:
# Tạo bản copy để không ảnh hưởng dữ liệu gốc
df = train_df.copy()
df_test = test_df.copy()

# ===== XÓA CỘT có quá nhiều missing (>70%) =====
threshold = 70
cols_to_drop = [col for col in df.columns if df[col].isnull().mean() * 100 > threshold]
print(f"Các cột có >{threshold}% missing (sẽ bị xóa): {cols_to_drop}")

df.drop(cols_to_drop, axis=1, inplace=True)
df_test.drop(cols_to_drop, axis=1, inplace=True)

print(f"\nShape sau khi xóa cột: Train {df.shape}, Test {df_test.shape}")

In [ ]:
# ===== XÓA CỘT KHÔNG CẦN THIẾT =====
# PassengerId: chỉ là ID, không mang thông tin dự đoán
# Ticket: phần lớn giá trị unique, khó sử dụng
# Name: sẽ xóa (có thể trích Title trước nếu cần)

# Lưu PassengerId của test để dùng cho submission sau này
test_passenger_ids = df_test['PassengerId'].copy()

df.drop(['PassengerId', 'Ticket', 'Name'], axis=1, inplace=True)
df_test.drop(['PassengerId', 'Ticket', 'Name'], axis=1, inplace=True)

print("Các cột còn lại:")
print(df.columns.tolist())
print(f"\nMissing còn lại:\n{df.isnull().sum()[df.isnull().sum() > 0]}")

## 6. Xử lý Missing Data - Imputation cơ bản (Mean/Median/Mode)

| Phương pháp | Khi nào dùng |
|---|---|
| **Mean** (trung bình) | Dữ liệu số, phân phối đều, ít outlier |
| **Median** (trung vị) | Dữ liệu số, có outlier (ít bị ảnh hưởng bởi giá trị cực đoan) |
| **Mode** (giá trị phổ biến nhất) | Dữ liệu categorical |

Cột `Age` có outlier → dùng **Median**.  
Cột `Embarked` là categorical → dùng **Mode**.  
Cột `Fare` (test) là số → dùng **Median**.

In [ ]:
# ===== CÁCH 1: Dùng sklearn SimpleImputer =====
# Impute Age bằng Median
age_imputer = SimpleImputer(strategy="median")
df["Age"] = age_imputer.fit_transform(df[["Age"]]).ravel()
df_test["Age"] = age_imputer.transform(df_test[["Age"]]).ravel()  # dùng transform (không fit lại) trên test

# Impute Embarked bằng Mode (most_frequent)
embarked_imputer = SimpleImputer(strategy="most_frequent")
df["Embarked"] = embarked_imputer.fit_transform(df[["Embarked"]]).ravel()
df_test["Embarked"] = embarked_imputer.transform(df_test[["Embarked"]]).ravel()

# Impute Fare (nếu thiếu trong test) bằng Median
fare_imputer = SimpleImputer(strategy="median")
fare_imputer.fit(df[["Fare"]])  # fit trên train
df_test["Fare"] = fare_imputer.transform(df_test[["Fare"]]).ravel()

print("Sau khi impute bằng SimpleImputer:")
print(f"  Train missing: {df.isnull().sum().sum()}")
print(f"  Test missing: {df_test.isnull().sum().sum()}")

In [ ]:
# ===== CÁCH 2 (thay thế): Dùng pandas fillna() =====
# Cách này đơn giản hơn, nhưng không lưu được imputer để áp dụng cho test

# df["Age"].fillna(df["Age"].median(), inplace=True)
# df["Embarked"].fillna(df["Embarked"].mode()[0], inplace=True)
# df_test["Fare"].fillna(df["Fare"].median(), inplace=True)

# So sánh kết quả
print("Giá trị impute được sử dụng:")
print(f"  Age median: {age_imputer.statistics_[0]}")
print(f"  Embarked mode: {embarked_imputer.statistics_[0]}")
print(f"  Fare median: {fare_imputer.statistics_[0]}")

## 7. Xử lý Missing Data - Imputation nâng cao (KNN, Iterative)

Các phương pháp nâng cao sử dụng **thông tin từ các cột khác** để dự đoán giá trị thiếu, thay vì chỉ dùng mean/median đơn giản.

| Phương pháp | Mô tả |
|---|---|
| **KNNImputer** | Dùng K hàng gần nhất (dựa trên các feature khác) để tính giá trị thiếu |
| **IterativeImputer** | Dùng model hồi quy lặp đi lặp lại để impute (tương tự MICE) |

> **Lưu ý:** Các phương pháp này chỉ hoạt động với dữ liệu số. Ta sẽ demo trên dữ liệu gốc (trước khi impute ở bước 6).

In [ ]:
# Demo so sánh các phương pháp impute Age trên dữ liệu gốc
df_demo = train_df[['Age', 'Pclass', 'SibSp', 'Parch', 'Fare']].copy()

# 1. SimpleImputer - Median
simple_imp = SimpleImputer(strategy='median')
age_simple = simple_imp.fit_transform(df_demo)[:, 0]

# 2. KNNImputer (dùng 5 hàng gần nhất)
knn_imp = KNNImputer(n_neighbors=5)
age_knn = knn_imp.fit_transform(df_demo)[:, 0]

# 3. IterativeImputer (dùng regression)
iter_imp = IterativeImputer(max_iter=10, random_state=42)
age_iter = iter_imp.fit_transform(df_demo)[:, 0]

# So sánh phân phối
fig, axes = plt.subplots(1, 4, figsize=(20, 4))

# Gốc (có missing)
train_df['Age'].dropna().hist(bins=30, ax=axes[0], color='gray', alpha=0.7)
axes[0].set_title('Gốc (chỉ non-null)')
axes[0].set_xlabel('Age')

# Median
pd.Series(age_simple).hist(bins=30, ax=axes[1], color='blue', alpha=0.7)
axes[1].set_title('Median Impute')
axes[1].set_xlabel('Age')

# KNN
pd.Series(age_knn).hist(bins=30, ax=axes[2], color='green', alpha=0.7)
axes[2].set_title('KNN Impute (k=5)')
axes[2].set_xlabel('Age')

# Iterative
pd.Series(age_iter).hist(bins=30, ax=axes[3], color='red', alpha=0.7)
axes[3].set_title('Iterative Impute')
axes[3].set_xlabel('Age')

plt.suptitle('So sánh các phương pháp Impute cột Age', fontsize=14)
plt.tight_layout()
plt.show()

print(f"Median Impute - Mean: {age_simple.mean():.2f}, Std: {age_simple.std():.2f}")
print(f"KNN Impute    - Mean: {age_knn.mean():.2f}, Std: {age_knn.std():.2f}")
print(f"Iter Impute   - Mean: {age_iter.mean():.2f}, Std: {age_iter.std():.2f}")

## 8. Mã hóa biến Categorical (Encoding)

| Phương pháp | Khi nào dùng | Ví dụ |
|---|---|---|
| **LabelEncoder** | Biến có thứ tự (ordinal) | Pclass: 1 < 2 < 3 |
| **OneHotEncoder / get_dummies** | Biến không có thứ tự (nominal) | Sex, Embarked |

In [ ]:
# ===== Feature Engineering trước khi encode =====
# Tạo FamilySize
df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
df_test['FamilySize'] = df_test['SibSp'] + df_test['Parch'] + 1

# Tạo AgeGroup
bins_age = [0, 12, 18, 30, 50, 200]
labels_age = ['Child', 'Teen', 'Young Adult', 'Adult', 'Senior']
df['AgeGroup'] = pd.cut(df['Age'], bins=bins_age, labels=labels_age)
df_test['AgeGroup'] = pd.cut(df_test['Age'], bins=bins_age, labels=labels_age)

# Tạo FareGroup
bins_fare = [0, 10, 30, 1000]
labels_fare = ['Low', 'Medium', 'High']
df['FareGroup'] = pd.cut(df['Fare'], bins=bins_fare, labels=labels_fare)
df_test['FareGroup'] = pd.cut(df_test['Fare'], bins=bins_fare, labels=labels_fare)

print("Các cột hiện tại:")
print(df.columns.tolist())
display(df.head())

In [ ]:
# ===== CÁCH 1: OneHotEncoder (sklearn) =====
cat_columns = ['Sex', 'Embarked', 'AgeGroup', 'FareGroup']

encoder = OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore')
encoder.fit(df[cat_columns])

# Transform train
encoded_train = pd.DataFrame(
    encoder.transform(df[cat_columns]),
    columns=encoder.get_feature_names_out(cat_columns),
    index=df.index
)
df_encoded = pd.concat([df.drop(cat_columns, axis=1), encoded_train], axis=1)

# Transform test
encoded_test = pd.DataFrame(
    encoder.transform(df_test[cat_columns]),
    columns=encoder.get_feature_names_out(cat_columns),
    index=df_test.index
)
df_test_encoded = pd.concat([df_test.drop(cat_columns, axis=1), encoded_test], axis=1)

print(f"Train shape sau encode: {df_encoded.shape}")
print(f"Test shape sau encode: {df_test_encoded.shape}")
display(df_encoded.head())

In [ ]:
# ===== CÁCH 2 (thay thế): pd.get_dummies() - đơn giản hơn =====
# df_encoded = pd.get_dummies(df, columns=['Sex', 'Embarked', 'AgeGroup', 'FareGroup'], drop_first=True)
# df_test_encoded = pd.get_dummies(df_test, columns=['Sex', 'Embarked', 'AgeGroup', 'FareGroup'], drop_first=True)

# ===== CÁCH 3: LabelEncoder (cho biến ordinal) =====
# Ví dụ: nếu muốn encode sex thành 0/1
le = LabelEncoder()
print("LabelEncoder demo trên cột Sex:")
print(f"  Classes: {le.fit(df['Sex']).classes_}")
print(f"  Encoded: female=0, male=1")

## 9. Feature Scaling (Chuẩn hóa & Normalize)

| Phương pháp | Công thức | Khi nào dùng |
|---|---|---|
| **StandardScaler** | $(x - \mu) / \sigma$ → mean=0, std=1 | Khi dùng SVM, Neural Network, Logistic Regression |
| **MinMaxScaler** | $(x - min) / (max - min)$ → [0, 1] | Khi cần giá trị trong khoảng cố định, KNN |

> **Quan trọng:** Fit scaler trên **train** rồi dùng `transform()` cho **test** (không fit lại trên test).

In [ ]:
# Tách features và target
X_train = df_encoded.drop('Survived', axis=1)
y_train = df_encoded['Survived']
X_test = df_test_encoded.copy()

# Các cột số cần scale
numerical_cols = ['Age', 'SibSp', 'Parch', 'Fare', 'FamilySize']

# ===== StandardScaler =====
scaler_std = StandardScaler()
X_train_std = X_train.copy()
X_test_std = X_test.copy()

X_train_std[numerical_cols] = scaler_std.fit_transform(X_train[numerical_cols])
X_test_std[numerical_cols] = scaler_std.transform(X_test[numerical_cols])

# ===== MinMaxScaler =====
scaler_mm = MinMaxScaler()
X_train_mm = X_train.copy()
X_test_mm = X_test.copy()

X_train_mm[numerical_cols] = scaler_mm.fit_transform(X_train[numerical_cols])
X_test_mm[numerical_cols] = scaler_mm.transform(X_test[numerical_cols])

# So sánh kết quả
print("===== Trước Scaling =====")
display(X_train[numerical_cols].describe().round(2))

print("\n===== Sau StandardScaler =====")
display(X_train_std[numerical_cols].describe().round(2))

print("\n===== Sau MinMaxScaler =====")
display(X_train_mm[numerical_cols].describe().round(2))

## 10. Kiểm tra Dữ liệu sau Xử lý

Xác nhận không còn missing values và data types đúng.

In [ ]:
# Sử dụng dữ liệu đã StandardScaler (phổ biến nhất)
print("===== KIỂM TRA TRAIN DATA =====")
print(f"Shape: {X_train_std.shape}")
print(f"Missing values: {X_train_std.isnull().sum().sum()}")
print(f"\nData types:\n{X_train_std.dtypes}")

print("\n===== KIỂM TRA TEST DATA =====")
print(f"Shape: {X_test_std.shape}")
print(f"Missing values: {X_test_std.isnull().sum().sum()}")

print("\n===== THỐNG KÊ MÔ TẢ =====")
display(X_train_std.describe().round(3))

## 11. Xuất Dữ liệu đã Xử lý

Lưu file CSV và download về máy (hoặc lưu lên Google Drive).

In [ ]:
# Lưu dữ liệu đã xử lý
X_train_std.to_csv('train_preprocessed.csv', index=False)
X_test_std.to_csv('test_preprocessed.csv', index=False)
y_train.to_csv('y_train.csv', index=False)

print("Đã lưu 3 file:")
print("  - train_preprocessed.csv")
print("  - test_preprocessed.csv")
print("  - y_train.csv")

# Download file về máy (chỉ chạy trên Colab)
from google.colab import files
files.download('train_preprocessed.csv')
files.download('test_preprocessed.csv')

# Hoặc lưu lên Google Drive:
# import shutil
# shutil.copy('train_preprocessed.csv', '/content/drive/MyDrive/')
# shutil.copy('test_preprocessed.csv', '/content/drive/MyDrive/')